# EXP04 — Avaliação histórica de janeiro de 2026

Este notebook faz parte do pacote de reprodução. Leia as explicações antes de executar. **Candidatos não verificados permanecem sem rótulo confirmado.**

## 1. Preparar o ambiente

No Colab, a primeira célula baixa uma cópia do projeto caso ela ainda não esteja disponível. No Python local, use o ambiente e as dependências descritos no README.

In [ ]:
from pathlib import Path
import os, subprocess, sys

def preparar_projeto():
    atual = Path.cwd()
    for pasta in [atual, *atual.parents]:
        if (pasta / "src" / "ethmix" / "__init__.py").exists():
            raiz = pasta
            break
    else:
        raiz = Path("/content/ethereum-mixer-reproducibility")
        if not raiz.exists():
            subprocess.run(["git", "clone", "--depth", "1",
                "https://github.com/BorgesGusta/ethereum-mixer-reproducibility.git",
                str(raiz)], check=True)
    sys.path.insert(0, str(raiz / "src"))
    os.chdir(raiz)
    return raiz

RAIZ = preparar_projeto()
print("Projeto:", RAIZ)


### Bibliotecas

Esta célula instala as bibliotecas necessárias na sessão atual, caso ainda não estejam presentes.

In [ ]:
import subprocess, sys
subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-r", "requirements.txt"], check=True)

## 2. Contexto

O arquivo processado de 15/01/2026 deve estar em `data/processed/features_2026-01-15.parquet`. A execução original empregou limiar 0,85 e registrou nove alertas, três pools conhecidos. **O experimento não é uma validação cega independente.**

In [ ]:
from ethmix.features import FEATURES
from ethmix.execucao import abrir_dados, treinar, prever

pasta = RAIZ / "data" / "processed"
treino = abrir_dados(pasta / "features_treino.parquet", ["label", "n_tx", *FEATURES])
dados_2026 = abrir_dados(pasta / "features_2026-01-15.parquet", ["recipient", "n_tx", *FEATURES])
modelo, _ = treinar(treino)
resultado = prever(modelo, dados_2026, min_tx=10, limiar=0.85)
print("Destinatários elegíveis:", len(resultado))
print("Alertas:", int(resultado.alerta.sum()))
display(resultado.loc[resultado.alerta, ["recipient", "score"]].sort_values("score", ascending=False))

## 3. Salvar sem atribuir rótulos

Os seis candidatos adicionais da execução histórica continuam sem classificação independente confirmada.

In [ ]:
saida = RAIZ / "results" / "generated" / "EXP04"
saida.mkdir(parents=True, exist_ok=True)
resultado.loc[resultado.alerta].to_csv(saida / "alertas_2026.csv", index=False)
print("Resultados salvos em", saida)